# N008 · 标准库算法工具箱

**目标：** 根据所需操作选择工具而非重复造轮子。

**先修：** N006, N007。本章自包含，依次运行即可；建议先读推导，再预测输出。

**知识点：** Counter/defaultdict/deque；heapq/bisect导览；cache；itertools；math。

**配套讲解来源：** [同名逐章意见](../../comment/008_standard_library_toolbox.md)。

本章阅读顺序：**先明白问题 → 补基础 → 手算 → 看核心片段 → 分段运行 → 核对测试**。需要复制使用时，跳到末尾“完整代码”，它包含全部依赖、实现和测试。

## 1. 通俗解释：输入什么，要得到什么

这一章要解决的问题可以用一句话概括：**当你需要一个"计数器"、"一个排队的小窗口"或者"一堆数里的最值"时，不要再自己手写循环造轮子了，Python 标准库里早就有人替你写好了，而且写得比你好。**

本章的"主菜"是一个具体问题：**找字符串里第一个只出现一次的字符的位置**。

给你一个具体到数字的例子：

- 输入：`s = "leetcode"`
- 我们逐个数一下每个字符出现几次：`l` 出现 1 次，`e` 出现 3 次，`t` 出现 1 次，`c` 出现 1 次，`o` 出现 1 次，`d` 出现 1 次。
- 从左往右看，第一个"只出现一次"的字符是下标 0 的 `l`。
- 所以输出：`0`。

再举一个找不到的例子：输入 `s = "aabb"`，`a` 出现 2 次、`b` 出现 2 次，没有任何字符只出现一次，所以输出 `-1`。

为什么答案是这样？因为我们要的是"第一个"（位置最靠左的）满足"唯一"（出现次数恰好为 1）这两个条件的字符位置。两个条件缺一不可：出现 1 次是"唯一"，从左往右第一次碰到就是"第一个"。

## 2. 基础知识：先读懂这些词

- **Counter（计数器）**：`collections.Counter` 是一个"专门用来数数"的字典。你把一个字符串或列表丢给它，它就自动帮你统计每个元素出现几次，比如 `Counter('ab')` 会得到 `{'a': 1, 'b': 1}`。我们用它是因为"数每个字符出现几次"是找唯一字符的前置步骤，手写循环容易错，Counter 一行搞定。
- **defaultdict（默认值字典）**：`collections.defaultdict` 是一种"你没写过的 key 也有默认值"的字典。本章知识点里提到它，典型用法是 `defaultdict(list)`——第一次访问某个 key 时自动给你一个空列表，省掉"先判断 key 在不在"的样板代码。
- **deque（双端队列）**：`collections.deque` 是"两头都能高效进出"的队列。它和列表的区别在于：列表在头部删一个元素，后面所有元素都要整体往前搬一格，代价是 O(n)；而 deque 的 `popleft()` 从头部取元素是 O(1)。我们用它是因为"排队"场景（先来先走）根本不需要碰中间。
- **heapq（堆）**：标准库里的堆模块，能让你随时快速拿到"当前最小值"。本章只是导览级别提一下，它适合"反复取最小/最大"的场景（比如前 K 大问题）。
- **bisect（二分插入/查找）**：在**已经排好序**的列表上，用二分查找快速定位"某个值应该插在哪"。前提是列表必须有序，否则结果没有意义。
- **cache（缓存装饰器）**：`functools.cache`（知识点中提及）能把函数算过的结果存起来，同样的参数再来一次就直接返回旧答案，是记忆化（memoization：把算过的答案存起来，下次直接查，不再重算）的现成工具。
- **itertools（迭代工具）**：标准库里的"循环拼装工具箱"，提供排列、组合、无限循环等现成迭代器。
- **math（数学模块）**：提供 gcd、sqrt、组合数等常用数学函数。
- **FIFO（先进先出）**：First In First Out，排队规则——谁先进队谁先出队。队列这种数据结构的核心性质就是它。
- **两遍扫描（two-pass）**：第一遍把信息收集完（这里是数频次），第二遍按题目要求（从左到右）利用这些信息找答案。这是本章代码的核心套路。

## 3. 思路推导：从小例子开始

我们把"找第一个唯一字符"拆成步骤：

- **Step 1：先想清楚"唯一"怎么判定。** 一个字符唯一，等价于它在整个字符串里恰好出现 1 次。所以我们必须先知道每个字符的总频次——只看前缀是判断不了的，比如 `"ee"` 的第一个 `e`，你得看到最后一个字符才知道它重复了。
- **Step 2：第一遍扫描，统计频次。** 用 `Counter(s)` 一句话把每个字符出现了几次全部数出来。这一遍是"收集情报"。
- **Step 3：第二遍扫描，从左到右找答案。** 拿着频次表重新走一遍字符串，走到每个字符时查一下它的频次，第一个查到频次等于 1 的位置，就是答案，立刻返回。因为是从左往右走的，所以第一次碰到的必然是"最左"的那个。
- **Step 4：兜底。** 如果第二遍走完了都没遇到频次为 1 的字符（比如空串或所有字符都重复），返回 `-1` 表示不存在。

**手算演示（输入 `"leetcode"`）：**

第一遍数频次，得到：

| 字符 | l | e | t | c | o | d |
|------|---|---|---|---|---|---|
| 次数 | 1 | 3 | 1 | 1 | 1 | 1 |

第二遍从左往右查表：

1. 下标 0 是 `l`，频次 1 —— 中了！返回 0。

就这么快。换个更能看出过程的例子 `"aabbc"`：

1. 第一遍数频次：`a:2, b:2, c:1`。
2. 下标 0 是 `a`，频次 2，跳过。
3. 下标 1 是 `a`，频次 2，跳过。
4. 下标 2 是 `b`，频次 2，跳过。
5. 下标 3 是 `b`，频次 2，跳过。
6. 下标 4 是 `c`，频次 1，返回 4。

**deque 的排队演示（对应 “运行示例”部分 的小实例表格）：** notebook 里对队列 `[4, 1, 7]` 做了操作追踪，表格逐行记录"操作、元素、队列状态"：

| 操作 | 元素 | 队列状态 |
|------|------|----------|
| append | 4 | [4] |
| append | 1 | [4, 1] |
| append | 7 | [4, 1, 7] |
| popleft | 4 | [1, 7] |
| popleft | 1 | [7] |
| popleft | 7 | [] |

你看：4 最早进队，也最早出队——这就是 FIFO。如果用普通列表的 `pop(0)` 也能做到同样的输出，但每次 `pop(0)` 都要把后面所有元素整体前移一格；deque 的 `popleft()` 没有这个搬移开销。

## 4. 核心代码：first_unique_char

先关注决定答案的状态与更新操作。这是完整实现的对应片段，不是另一套解法；导入、辅助结构及其余接口在下一节补全。

```python
def first_unique_char(s: str) -> int:
    counts = Counter(s)
    for i, c in enumerate(s):
        if counts[c] == 1:
            return i
    return -1
```

### 逐段读懂代码

“分段实现”部分 里只有两个函数，我们逐个看。

### 函数 1：`first_unique_char` —— 找第一个唯一字符的下标

```python
from collections import Counter, deque

def first_unique_char(s: str) -> int:
    counts = Counter(s)
    for i, c in enumerate(s):
        if counts[c] == 1:
            return i
    return -1
```

- `from collections import Counter, deque`：从标准库 collections 包里把计数器和双端队列引进来。这两个都是 Python 自带的，不需要安装任何东西。
- `counts=Counter(s)`：把整个字符串喂给 Counter，它内部遍历一遍 `s`，返回一个"字符 → 出现次数"的字典。比如 `Counter('leetcode')` 得到 `{'l':1,'e':3,'t':1,'c':1,'o':1,'d':1}`。这就是第一遍扫描，一句话完成。
- `for i,c in enumerate(s)`：第二遍扫描。`enumerate` 会同时给我们下标 `i` 和字符 `c`，这样我们既知道"当前是哪个字符"，又知道"它在第几位"，两者都是返回答案需要的。
- `if counts[c]==1: return i`：查频次表，当前字符恰好出现 1 次，说明它唯一；因为我们是按从左到右的顺序走的，第一个命中处就是最左答案，直接返回下标。
- `return -1`：循环跑完都没命中，说明不存在唯一字符（包括空字符串这种极端情况——循环一次都不进，直接落到这里），按约定返回 `-1`。

### 函数 2：`deque_queue_demo` —— 用 deque 演示队列

```python
def deque_queue_demo(items):
    queue = deque(items)
    result = []
    while queue:
        result.append(queue.popleft())
    return result
```

- `queue=deque(items)`：把传入的序列（比如 `[3,1,2]`）装进双端队列，队列内容是 `deque([3, 1, 2])`。
- `result=[]`：准备一个空列表收集出队顺序。
- `while queue:`：只要队列里还有元素就继续。注意空队列 `deque([])` 在布尔判断里是假值，所以队列空了循环自动停，不会多弹也不会报错。
- `result.append(queue.popleft())`：每次从**左边**（队头）取一个元素放进结果。左边进来的最早，也最早被取走，所以整体效果是 FIFO。
- `return result`：返回出队顺序。对 `[3,1,2]` 来说，3 先进先出，然后 1，然后 2，结果是 `[3,1,2]`——进队什么顺序，出队就什么顺序，这正是队列的定义。

另外 “运行示例”部分 还有个 `show_table` 辅助函数，它只做展示：把"操作/元素/队列状态"逐行拼成 HTML 表格，第三节里的演示表就是它生成的。它不参与算法本身，你可以把它当成一个"把过程画给你看"的小黑板。

## 5. 分段实现：按顺序运行

**本章接口：** `first_unique_char(s)`、`deque_queue_demo(items)`

### 导入本章使用的库

In [1]:
from collections import Counter, deque

### first_unique_char

In [2]:
def first_unique_char(s: str) -> int:
    counts = Counter(s)
    for i, c in enumerate(s):
        if counts[c] == 1:
            return i
    return -1

### deque_queue_demo

In [3]:
def deque_queue_demo(items):
    queue = deque(items)
    result = []
    while queue:
        result.append(queue.popleft())
    return result

## 6. 运行示例与状态可视化

先预测结果再运行。手算表帮助解释过程；下面的表由当前实现计算。若表只展示输入输出，它并不代替内部状态轨迹。

In [4]:
from IPython.display import display, HTML
from html import escape

def show_table(headers, rows):
    """Render a small, explicit state trace; no network or hidden algorithm code."""
    head = ''.join('<th>' + escape(str(x)) + '</th>' for x in headers)
    body = ''.join('<tr>' + ''.join('<td>' + escape(str(x)) + '</td>' for x in row) + '</tr>' for row in rows)
    display(HTML('<table><thead><tr>' + head + '</tr></thead><tbody>' + body + '</tbody></table>'))

q=deque()
rows=[]
for x in [4,1,7]:
    q.append(x); rows.append(('append',x,list(q)))
while q:
    x=q.popleft(); rows.append(('popleft',x,list(q)))
show_table(['操作','元素','队列状态'],rows)

操作,元素,队列状态
append,4,[4]
append,1,"[4, 1]"
append,7,"[4, 1, 7]"
popleft,4,"[1, 7]"
popleft,1,[7]
popleft,7,[]


## 7. 正确性、适用条件与复杂度

首遍得到完整频次；第二遍从左至右扫描，首次频次为 1 的位置必然是最左合法答案。deque按一端入、一端出保持FIFO。

**代价：** 两个实现均为 O(n) 时间，分别需 O(u) 计数空间和 O(n) 队列/输出空间。

### 展开理解

**为什么 `first_unique_char` 是对的？** 我们分两半说。第一半，`Counter(s)` 确实数出了每个字符的真实总次数，因为字符串里每个字符它都看过了、都加了 1，没有遗漏也没有多数。第二半，第二遍循环是按下标从小到大走的，所以碰到的**第一个**"频次为 1"的字符，一定就是所有唯一字符里位置最靠左的那个——不可能有更靠左的唯一字符了，因为如果有，我们早就在更小的下标处碰到并返回了。如果整个循环都没碰到频次为 1 的字符，那答案确实不存在，返回 `-1` 也是对的。

**为什么 deque 保持先进先出？** 因为元素只从右边 `append` 进去、只从左边 `popleft` 出来，两头各管一边。进来越早的元素离左出口越近，所以越早出去。谁也插不了队。

**复杂度是多少？** 两个函数都是**线性时间 O(n)**，n 是输入长度。具体到 `first_unique_char`：Counter 内部扫一遍字符串是 n 次计数操作，第二遍循环最多再扫 n 次，总共大约 2n 次基本操作。给你一个直观量级：如果字符串有 10⁵（十万）个字符，2n 就是二十万次操作，现代计算机一秒能做上亿次基本操作，所以这一瞬间就算完了。空间方面：频次表只存"出现过的不同字符"，设不同字符数是 u，就占 O(u)；比如十万个字符全是 26 个小写字母，表里也只有 26 项。`deque_queue_demo` 的队列和输出列表各装 n 个元素，所以空间是 O(n)。

## 8. 单元测试：每个用例在检查什么

“自动测试”部分 的每条 assert 都在检查一个明确的契约：

- `assert first_unique_char('leetcode') == 0`：**正常值测试**。`'leetcode'` 里 `l` 是首个唯一字符，下标 0。这条验证"能找到答案且返回的是最左位置"。
- `assert first_unique_char('aabb') == -1`：**无解测试**。所有字符都出现两次，不存在唯一字符，必须返回 -1 而不是随便一个下标。它防止实现"找不到时返回错误值"。
- `assert first_unique_char('') == -1`：**边界值测试（空输入）**。空字符串一个字符都没有，第二遍循环一次都不执行，代码必须稳妥地落到 `return -1`。这条专门防空输入崩溃。
- `assert deque_queue_demo([3,1,2]) == [3,1,2]`：**正常值 + FIFO 性质测试**。出队顺序必须和进队顺序完全一致，验证 deque 的排队行为没被打乱。
- `c=Counter('ab'); before=dict(c); assert c['z']==0; assert dict(c)==before`：**特殊行为测试**。它检查的是 Counter 一个容易被忽视的性质：查询一个从未出现过的 key（比如 `'z'`）会返回 0 而不是抛 KeyError；而且这次查询是"只读"的——查询之后计数器的内容和查询之前一模一样（`before` 快照比对）。这条防止你以为"查一下会把 z 计成 0 加进表里"。

最后一行 `print('N008: 所有本章断言通过')` 只是把通过信息打出来，方便你肉眼确认整段测试都跑过了。

In [5]:
assert first_unique_char('leetcode') == 0

assert first_unique_char('aabb') == -1

assert first_unique_char('') == -1

assert deque_queue_demo([3, 1, 2]) == [3, 1, 2]

c = Counter('ab')

before = dict(c)

assert c['z'] == 0

assert dict(c) == before

print('N008: 所有本章断言通过')

N008: 所有本章断言通过


## 9. 练习：先尝试，再看提示

**练习 1：** 把手写计数改为Counter。

**练习 2：** 写出每种工具适用与不适用操作。

<details>
<summary>展开思路提示（不是完整答案）</summary>

“练习”部分 给了两道练习，方向提示如下：

- **练习 1（把手写计数改成 Counter）**：先别急着写代码。你在 N007（字典）那章大概手写过"遍历字符串、判断 key 在不在、不在就置 0 加 1"的计数循环。这道练习就是把它压缩成 `Counter(s)` 一行。提示分三步走：第一，写清楚输入输出（输入是字符串，输出是频次字典）；第二，先手算一个小例子，比如 `"aab"` 应得 `{'a':2,'b':1}`；第三，分别用手写版和 Counter 版跑同一个输入，比对两个字典是否相等——相等就说明替换是对的。边界别忘了空字符串：两个版本都应得到空字典。
- **练习 2（每种工具的适用与不适用）**：做一个"工具选型表"。对每个工具问两个问题：它哪个操作是 O(1) 级别的快？哪个操作会退化成 O(n) 级别的慢？给你两个起点提示：列表的 `pop(0)` 是慢操作（整体搬移），换成 deque 的 `popleft` 就快了；bisect 在无序列表上跑，结果是错的，它的前提就是"必须先有序"。同样，每写一行结论都配一个小例子手算验证，别凭印象写。

</details>

## 10. 完整代码：可独立运行

下面这一格包含**全部导入、全部接口和验证用例**，不依赖前面的单元格。可以重启内核后只运行这一格，也可以复制到 `.py` 文件后运行。在 Notebook 中 Run All 时会再验证一次，这是为独立复制使用保留的完整版本。

In [6]:
# 完整实现：本单元格不依赖其他单元格。
from collections import Counter, deque

def first_unique_char(s: str) -> int:
    counts = Counter(s)
    for i, c in enumerate(s):
        if counts[c] == 1:
            return i
    return -1

def deque_queue_demo(items):
    queue = deque(items)
    result = []
    while queue:
        result.append(queue.popleft())
    return result

# 示例与回归测试
assert first_unique_char('leetcode') == 0

assert first_unique_char('aabb') == -1

assert first_unique_char('') == -1

assert deque_queue_demo([3, 1, 2]) == [3, 1, 2]

c = Counter('ab')

before = dict(c)

assert c['z'] == 0

assert dict(c) == before

print('N008: 所有本章断言通过')

N008: 所有本章断言通过


## 11. 代表题与迁移

“题目映射”部分 列出的题目：

- **387. First Unique Character in a String（字符串中的第一个唯一字符）**：这题就是本章 `first_unique_char` 的原题——练的是"Counter 统计频次 + 两遍扫描取最左答案"这个组合拳，属于热身级别（warmup）。

最后提醒一句（也是 notebook 原文强调的）：这里的实现遵循的是教学契约，能覆盖该题的核心思路，但刷题时仍应以 LeetCode 官方题面和评测为准。

**题面入口：** 

题目用于知识映射；本章完整代码遵循教学接口，不等于每道题的完整平台提交。

## 12. 掌握检查

合上代码，说明输入输出、核心变量的含义、每次更新为什么成立，以及一个边界情况。再完成一道练习，不能只凭“运行通过”判断掌握。

**继续学习：** [课程索引](../../COURSE_INDEX.md)。